# Is the 20% lost-exporter threshold a meaningful discontinuity?

## A simple regression-discontinuity-style diagnostic

The existing rule calls a country–product loss *large* when SRCA falls by at least 20% while crossing from $SRCA_t \geq 1$ to $SRCA_{t+1}<1$. This notebook asks a narrow question: **do outcomes after the crossing change discontinuously at a 20% decline?**

Among all threshold-crossing country–product pairs:

- Running variable: $R=1-SRCA_{t+1}/SRCA_t$
- Cutoff: $c=0.20$
- Indicator: $D=1[R\geq c]$
- Primary outcome: still below $SRCA=1$ at $t+2$
- Secondary outcome: cumulative log export change from $t$ to $t+2$

A local linear model is fitted separately on either side using triangular weights. This follows the practical local-linear approach described by [Imbens and Lemieux](https://www.nber.org/papers/t0337).

> **Interpretation limit:** the 20% rule is our classification rule; it does not assign an external treatment. Consequently, this is not a causal RDD. It is a threshold-validity diagnostic: a jump would support the idea that 20% separates different post-loss dynamics, while no jump would suggest that 20% is a convenient but arbitrary cutoff.

## 1. Setup

In [ ]:
from pathlib import Path
import math
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)

DIGITS = 4
CUTOFF = 0.20
BANDWIDTH = 0.10  # use declines from 10% to 30% in the main specification

def find_repo_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

ROOT = find_repo_root()
GRAPH_DIR = ROOT / "data" / "5. Graphs Data" / f"{DIGITS}_digits"
print(f"Repository: {ROOT}")

## 2. Construct the RDD sample

Only loss events with an observable $t+2$ are retained. The analysis therefore covers transitions through 2020→2021; the 2021→2022 losses cannot yet have a one-year-ahead outcome in these data.

In [ ]:
with open(GRAPH_DIR / "SRCA.pickle", "rb") as handle:
    srca = pickle.load(handle)
with open(GRAPH_DIR / "CPM.pickle", "rb") as handle:
    cpm_raw = pickle.load(handle)

years = sorted(srca)
country_index = srca[years[0]].index
product_columns = srca[years[0]].columns.astype(str)

exports = {}
for year, matrix in cpm_raw.items():
    matrix = matrix.set_index("country_id") if "country_id" in matrix.columns else matrix.copy()
    matrix.columns = matrix.columns.astype(str)
    exports[year] = matrix.reindex(index=country_index, columns=product_columns, fill_value=0).fillna(0)

rows = []
for t in years:
    if t + 2 not in srca:
        continue
    crossing = (srca[t] >= 1) & (srca[t + 1] < 1)
    country_pos, product_pos = np.where(crossing.to_numpy())
    for i, j in zip(country_pos, product_pos):
        srca_t = float(srca[t].iat[i, j])
        srca_t1 = float(srca[t + 1].iat[i, j])
        srca_t2 = float(srca[t + 2].iat[i, j])
        export_t = float(exports[t].iat[i, j])
        export_t2 = float(exports[t + 2].iat[i, j])
        rows.append({
            "country_id": country_index[i], "product_code": product_columns[j],
            "year_t": t, "srca_t": srca_t, "srca_t1": srca_t1, "srca_t2": srca_t2,
            "drop_pct": 1 - srca_t1 / srca_t,
            "persistent_loss_t2": float(srca_t2 < 1),
            "cumulative_export_change_t2": np.log1p(export_t2) - np.log1p(export_t),
            "baseline_log_exports": np.log1p(export_t)
        })

events = pd.DataFrame(rows)
events["above_20pct"] = (events["drop_pct"] >= CUTOFF).astype(int)
print(f"RDD sample: {len(events):,} country–product losses ({events.year_t.min()}–{events.year_t.max()} transitions)")
display(events[["drop_pct", "persistent_loss_t2", "cumulative_export_change_t2"]].describe())

## 3. Inspect the running variable

The cutoff must have observations on both sides. A smooth density is desirable. Here countries cannot precisely manipulate the statistic around our unpublished analytical threshold, so the density plot is mainly a data-quality check rather than a behavioral sorting test.

In [ ]:
local = events[events["drop_pct"].between(CUTOFF - BANDWIDTH, CUTOFF + BANDWIDTH)]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(local["drop_pct"], bins=40, color="#5d6d7e", edgecolor="white")
ax.axvline(CUTOFF, color="#c0392b", lw=2, label="20% cutoff")
ax.set(title="Distribution of SRCA declines near the proposed cutoff",
       xlabel="One-year SRCA decline", ylabel="Country–product events")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Within ±{BANDWIDTH:.0%}: {(local.drop_pct < CUTOFF).sum():,} below and {(local.drop_pct >= CUTOFF).sum():,} above the cutoff")

## 4. Fit the local linear specification

For observations with $|R-c|\leq h$, estimate

$$Y=\alpha+\tau D+\beta(R-c)+\gamma D(R-c)+\varepsilon.$$

$\tau$ is the estimated jump at 20%. Triangular weights give observations closest to the cutoff the most influence. Standard errors are clustered by country because each country contributes many products and years. The binary persistence outcome is fitted as a linear probability model to keep the approach transparent.

In [ ]:
def local_linear_rdd(data, outcome, cutoff=CUTOFF, bandwidth=BANDWIDTH, cluster="country_id"):
    sample = data.loc[(data["drop_pct"] - cutoff).abs() <= bandwidth].copy().reset_index(drop=True)
    x = sample["drop_pct"].to_numpy() - cutoff
    treated = (x >= 0).astype(float)
    design = np.column_stack([np.ones(len(sample)), treated, x, treated * x])
    weights = 1 - np.abs(x) / bandwidth
    outcome_values = sample[outcome].to_numpy(dtype=float)

    xtw = design.T * weights
    bread = np.linalg.inv(xtw @ design)
    coefficients = bread @ (xtw @ outcome_values)
    residuals = outcome_values - design @ coefficients

    # Country-clustered sandwich covariance.
    meat = np.zeros((design.shape[1], design.shape[1]))
    groups = sample[cluster].to_numpy()
    for group in np.unique(groups):
        keep = groups == group
        score = (design[keep] * (weights[keep] * residuals[keep])[:, None]).sum(axis=0)
        meat += np.outer(score, score)
    n, k, n_groups = len(sample), design.shape[1], np.unique(groups).size
    correction = (n_groups / (n_groups - 1)) * ((n - 1) / (n - k))
    covariance = correction * bread @ meat @ bread
    standard_errors = np.sqrt(np.diag(covariance))

    estimate, standard_error = coefficients[1], standard_errors[1]
    result = {
        "outcome": outcome, "cutoff": cutoff, "bandwidth": bandwidth,
        "estimate_tau": estimate, "clustered_se": standard_error,
        "ci_low": estimate - 1.96 * standard_error, "ci_high": estimate + 1.96 * standard_error,
        "normal_p_value": math.erfc(abs(estimate / standard_error) / np.sqrt(2)),
        "n_below": int((treated == 0).sum()), "n_above": int((treated == 1).sum()),
        "n_total": n
    }
    return result, coefficients, sample

primary, primary_coef, primary_sample = local_linear_rdd(events, "persistent_loss_t2")
secondary, _, _ = local_linear_rdd(events, "cumulative_export_change_t2")
results = pd.DataFrame([primary, secondary])
display(results.style.format({"cutoff": "{:.0%}", "bandwidth": "{:.0%}",
                              "estimate_tau": "{:.3f}", "clustered_se": "{:.3f}",
                              "ci_low": "{:.3f}", "ci_high": "{:.3f}",
                              "normal_p_value": "{:.3f}"}))

### Reading the primary estimate

Because the primary outcome is 0/1, $\tau=0.05$ means a 5-percentage-point upward jump in the probability of remaining below one. If its 95% interval includes zero, the data do not show a discontinuity at 20%; this is evidence against treating exactly 20% as a special empirical boundary, not proof that smaller losses are harmless.

In [ ]:
def binned_rdd_plot(sample, coefficients, outcome, cutoff=CUTOFF, bandwidth=BANDWIDTH, bins_each_side=12):
    fig, ax = plt.subplots(figsize=(10, 5))
    for side, color in [("Below 20%", "#2e86c1"), ("At least 20%", "#c0392b")]:
        below = side == "Below 20%"
        part = sample[(sample["drop_pct"] < cutoff) == below].copy()
        edges = np.linspace(part["drop_pct"].min(), part["drop_pct"].max(), bins_each_side + 1)
        part["bin"] = pd.cut(part["drop_pct"], edges, include_lowest=True)
        means = part.groupby("bin", observed=True).agg(x=("drop_pct", "mean"), y=(outcome, "mean"))
        ax.scatter(means["x"], means["y"], color=color, s=35, alpha=.8, label=f"{side}: binned means")

    left_x = np.linspace(cutoff - bandwidth, cutoff, 100, endpoint=False)
    right_x = np.linspace(cutoff, cutoff + bandwidth, 100)
    left_centered, right_centered = left_x - cutoff, right_x - cutoff
    ax.plot(left_x, coefficients[0] + coefficients[2] * left_centered, color="#2e86c1", lw=2.5)
    ax.plot(right_x, coefficients[0] + coefficients[1] +
            (coefficients[2] + coefficients[3]) * right_centered, color="#c0392b", lw=2.5)
    ax.axvline(cutoff, color="black", ls="--", lw=1)
    ax.set(title="RDD diagnostic at the 20% SRCA-decline cutoff", xlabel="One-year SRCA decline",
           ylabel="Probability SRCA remains below 1 at t+2")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    ax.yaxis.set_major_formatter(lambda y, pos: f"{y:.0%}")
    ax.legend(frameon=True); plt.tight_layout(); plt.show()

binned_rdd_plot(primary_sample, primary_coef, "persistent_loss_t2")

## 5. Bandwidth sensitivity

Narrow windows improve local comparability but reduce precision. The sign and approximate size of the jump should not change dramatically across reasonable bandwidths.

In [ ]:
bandwidth_results = []
for bandwidth in [0.05, 0.075, 0.10, 0.15]:
    result, _, _ = local_linear_rdd(events, "persistent_loss_t2", bandwidth=bandwidth)
    bandwidth_results.append(result)
bandwidth_results = pd.DataFrame(bandwidth_results)
display(bandwidth_results[["bandwidth", "estimate_tau", "clustered_se", "ci_low", "ci_high", "n_total"]]
        .style.format({"bandwidth": "{:.1%}", "estimate_tau": "{:.3f}",
                       "clustered_se": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}"}))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(100 * bandwidth_results["bandwidth"], bandwidth_results["estimate_tau"],
            yerr=1.96 * bandwidth_results["clustered_se"], fmt="o-", capsize=4, color="#7d3c98")
ax.axhline(0, color="black", lw=1)
ax.set(title="Sensitivity of the estimated persistence jump", xlabel="Bandwidth (percentage points)",
       ylabel="Estimated jump at 20%")
ax.yaxis.set_major_formatter(lambda y, pos: f"{y:.0%}")
plt.tight_layout(); plt.show()

## 6. Two simple falsification checks

1. **Predetermined covariates:** baseline SRCA and exports should not jump at 20%. A jump would indicate that locally different observations are being compared.
2. **Placebo cutoffs:** if similar jumps appear at many arbitrary cutoffs, 20% is not distinctive. Placebos are descriptive because their windows overlap.

In [ ]:
validity_rows = []
for outcome in ["srca_t", "baseline_log_exports"]:
    result, _, _ = local_linear_rdd(events, outcome)
    validity_rows.append(result)
print("Predetermined-covariate continuity checks")
display(pd.DataFrame(validity_rows)[["outcome", "estimate_tau", "clustered_se", "ci_low", "ci_high"]]
        .style.format({c: "{:.3f}" for c in ["estimate_tau", "clustered_se", "ci_low", "ci_high"]}))

placebo_rows = []
for cutoff in [0.10, 0.15, 0.20, 0.25, 0.30]:
    result, _, _ = local_linear_rdd(events, "persistent_loss_t2", cutoff=cutoff, bandwidth=0.10)
    placebo_rows.append(result)
print("Cutoff-placebo checks")
display(pd.DataFrame(placebo_rows)[["cutoff", "estimate_tau", "clustered_se", "ci_low", "ci_high", "n_total"]]
        .style.format({"cutoff": "{:.0%}", "estimate_tau": "{:.3f}", "clustered_se": "{:.3f}",
                       "ci_low": "{:.3f}", "ci_high": "{:.3f}"}))

## 7. Conclusion template

Use the following wording, replacing the values with the main result above:

> Among lost-exporter events within ±10 percentage points of the proposed 20% SRCA-decline threshold, a local linear threshold diagnostic estimated a **[τ × 100] percentage-point** change in the probability of remaining below SRCA = 1 one year later (95% CI: **[lower × 100, upper × 100]**). [Because the interval includes zero / Because the interval excludes zero], the data [do not show / show] a discrete change in post-loss persistence at 20%. This result [does not support / supports] treating 20% as a distinct empirical regime boundary, although it does not establish that the decline was caused by an external shock.

If there is no discontinuity, keep 20% only as a transparent sensitivity parameter and report results for several thresholds (for example 10%, 20%, and 30%). If a discontinuity appears, it is still necessary to link events to external shock dates before making a causal claim.

In [ ]:
tau, low, high = primary["estimate_tau"], primary["ci_low"], primary["ci_high"]
decision = "does not show" if low <= 0 <= high else "shows"
support = "does not support" if low <= 0 <= high else "supports"
print(
    f"Main result: τ = {100*tau:.1f} percentage points "
    f"(95% CI {100*low:.1f} to {100*high:.1f}; n = {primary['n_total']:,}).\n"
    f"The diagnostic {decision} a discontinuity in persistence at 20% and therefore {support} "
    "interpreting exactly 20% as a distinct empirical regime boundary."
)